In [1]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../data/raw/airtel.db')  
 
def q(sql):                                     

    return pd.read_sql(sql, con)

In [2]:
q(""" SELECT * FROM CIRCLES """)

,circle_id,circle_name,region
0,1,Delhi,North
1,2,Punjab,North
2,3,Haryana,North
3,4,Himachal Pradesh,North
4,5,Jammu & Kashmir,North
5,6,Rajasthan,North
6,7,Uttar Pradesh (East),North
7,8,Uttar Pradesh (West),North
8,9,Mumbai,West
9,10,Maharashtra,West


In [3]:
q("""
SELECT COUNT(*) AS total_count,
       COUNT(DISTINCT circle_id) AS distinct_count
FROM circles;
""")


# plans.plan_id
# customers.customer_id
# usage_daily.usage_id
# recharges.recharge_id
# complaints.complaint_id

,total_count,distinct_count
0,22,22


In [4]:
q("""
SELECT COUNT(*) AS total_count,
       COUNT(DISTINCT customer_id) AS distinct_count
FROM customers;
""")


# plans.plan_id
# customers.customer_id
# usage_daily.usage_id
# recharges.recharge_id
# complaints.complaint_id

,total_count,distinct_count
0,22000,22000


In [5]:
q("""
SELECT COUNT(*) AS total_count,
       COUNT(DISTINCT usage_id) AS distinct_count
FROM  usage_daily;
""")


# plans.plan_id
# customers.customer_id
# usage_daily.usage_id
# recharges.recharge_id
# complaints.complaint_id

,total_count,distinct_count
0,1326739,1326739


In [6]:
q("""
SELECT COUNT(*) AS total_count,
       COUNT(DISTINCT complaint_id) AS distinct_count
FROM  complaints;
""")


# plans.plan_id
# customers.customer_id
# usage_daily.usage_id
# recharges.recharge_id
# complaints.complaint_id

,total_count,distinct_count
0,26841,26841


In [7]:
q("""
SELECT COUNT(*)
FROM circles c
JOIN customers cu
ON cu.circle_id = c.circle_id;
""")

,COUNT(*)
0,22000


In [8]:
q("""
SELECT COUNT(*)
FROM circles c
Left JOIN customers cu
ON cu.circle_id = c.circle_id;
""")

,COUNT(*)
0,22000


In [9]:
# 2. Plans -> Customers
q("""
SELECT COUNT(*)
FROM plans p
JOIN customers cu
ON cu.customer_id = p.plan_id;
""")

,COUNT(*)
0,14


In [10]:
# 2. Plans -> Customers
q("""
SELECT COUNT(*)
FROM plans p
Left JOIN customers cu
ON cu.customer_id = p.plan_id;
""")

,COUNT(*)
0,14


In [11]:
# 3. Customers -> Recharges
q("""
SELECT COUNT(*)
FROM customers cu
JOIN recharges r
ON cu.customer_id = r.customer_id;
""")

,COUNT(*)
0,408089


In [12]:
# 3. Customers -> Recharges
q("""
SELECT COUNT(*)
FROM customers cu
Left JOIN recharges r
ON cu.customer_id = r.customer_id;
""")

,COUNT(*)
0,408089


In [13]:
# 4. Plans -> Recharges
q("""
SELECT COUNT(*)
FROM plans p
JOIN recharges r
ON p.plan_id = r.plan_id;
""")

,COUNT(*)
0,408089


In [14]:
# 4. Plans -> Recharges
q("""
SELECT COUNT(*)
FROM plans p
Left JOIN recharges r
ON p.plan_id = r.plan_id;
""")

,COUNT(*)
0,408089


In [15]:
# 5. Customers -> Usage Daily
q("""
SELECT COUNT(*)
FROM customers cu
JOIN usage_daily ud
ON cu.customer_id = ud.customer_id;
""")

,COUNT(*)
0,1326739


In [16]:
# 5. Customers -> Usage Daily
q("""
SELECT COUNT(*)
FROM customers cu
Left JOIN usage_daily ud
ON cu.customer_id = ud.customer_id;
""")    

,COUNT(*)
0,1326743


In [17]:
# 6. Customers -> Complaints
q("""
SELECT COUNT(*)
FROM customers cu
JOIN complaints co
ON cu.customer_id = co.customer_id;
""")

,COUNT(*)
0,26841


In [18]:
# 6. Customers -> Complaints
q("""
SELECT COUNT(*)
FROM customers cu
Left JOIN complaints co
ON cu.customer_id = co.customer_id;
""")

,COUNT(*)
0,33472


In [19]:
q("""
SELECT SUM(is_churned = 1 AND churn_date IS NOT NULL) 
            AS left_dated,
            SUM(is_churned = 0 AND churn_date IS NULL) 
            AS stayed_blank
     FROM customers;
""")

,left_dated,stayed_blank
0,5226,16774


In [20]:
q("""
SELECT SUM(is_resolved = 1 AND resolution_days IS NOT NULL) 
            AS left_dated,
            SUM(is_resolved = 0 AND resolution_days IS NULL) 
            AS stayed_blank
     FROM complaints;
""")

,left_dated,stayed_blank
0,18820,8021


In [21]:
q("""select c.customer_id,c.connection_type,
(select count(*)from recharges r
where r.customer_id = c.customer_id)
as payments
from customers c
where not exists (select 1 from usage_daily u where u.customer_id = c.customer_id);""")

,customer_id,connection_type,payments
0,266,Prepaid,1
1,10684,Prepaid,29
2,13508,Prepaid,8
3,19111,Prepaid,39


In [23]:
q("""

  WITH l AS (SELECT customer_id,     
             MAX(usage_date) AS last_use         
           FROM usage_daily GROUP BY customer_id)
SELECT c.is_churned, COUNT(*) AS subs, 
       ROUND(AVG(julianday('2025-06-30')
             - julianday(l.last_use)), 2)     
         AS avg_quiet,                    
       MIN(julianday('2025-06-30')              
             - julianday(l.last_use)) AS min_,
       MAX(julianday('2025-06-30')              
             - julianday(l.last_use)) AS max_   
FROM   customers c JOIN l                
       ON l.customer_id = c.customer_id     
GROUP  BY c.is_churned;

""")

,is_churned,subs,avg_quiet,min_,max_
0,0,16772,2.66,0.0,143.0
1,1,5224,87.80,7.0,180.0


In [25]:
q("""
SELECT 'usage_daily.usage_date' AS col,        
       MIN(usage_date) AS first_,                
       MAX(usage_date) AS last_                
FROM   usage_daily                    
UNION ALL SELECT 'customers.activation_date',   
       MIN(activation_date), MAX(activation_date)
FROM   customers                              
UNION ALL SELECT 'customers.churn_date',
       MIN(churn_date), MAX(churn_date) 
FROM   customers                      
UNION ALL SELECT 'recharges.recharge_date', 
       MIN(recharge_date), MAX(recharge_date)
FROM   recharges                           
UNION ALL SELECT 'complaints.complaint_date',
       MIN(complaint_date), MAX(complaint_date)
FROM   complaints;
""")

,col,first_,last_
0,usage_daily.usage_date,2025-01-01,2025-06-30
1,customers.activation_date,2022-01-01,2025-02-28
2,customers.churn_date,2025-01-16,2025-06-23
3,recharges.recharge_date,2022-01-01,2025-06-27
4,complaints.complaint_date,2025-01-01,2025-06-30


In [ ]:
q("""
WITH t AS (SELECT customer_id,                
             SUM(data_mb) AS total,           
             COUNT(*) AS days_active         
           FROM usage_daily GROUP BY customer_id) 
SELECT ROUND(AVG(total / days_active), 2)      
         AS per_active_day,                    
       ROUND(AVG(total / 181.0), 2)              
         AS per_calendar_day                
FROM   t;
""")

,per_active_day,per_calendar_day
0,1404.44,524.15
